# Pipeline Output Inspection
Quick QA notebook — load each parquet output and verify it looks correct.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import colorcet as cc

DATA_DIR = Path("data/")

# CONUS extent
CONUS_EXTENT = [-125, -66, 24, 50]  # [lon_min, lon_max, lat_min, lat_max]

def make_conus_ax(fig, pos=111, title=''):
    """Return a cartopy GeoAxes set to CONUS with standard features."""
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,       facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,      facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,      facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,     linewidth=0.4,       zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE,  linewidth=0.6,       zorder=3)
    ax.add_feature(cfeature.BORDERS,    linewidth=0.6,       zorder=3)
    if title:
        ax.set_title(title)
    return ax

## 1. Site Metadata

In [ ]:
site_info = pd.read_parquet(DATA_DIR / "metadata" / "site_info.parquet")
print(f"Shape: {site_info.shape}")
print(site_info.dtypes)
site_info

## 2. Data Coverage Summary

In [ ]:
# coverage = pd.read_parquet(DATA_DIR / "metadata" / "data_coverage.parquet")
# display(coverage)

# bool_cols = [c for c in coverage.columns if c != "site_no"]
# missing = coverage[bool_cols].eq(False).any(axis=1)
# if missing.any():
#     print("\nSites with missing data:")
#     display(coverage[missing])

## 2b. Coverage Map

Color coding by data tier:
- **Green** — stage + flood stage thresholds (full coverage)
- **Orange** — stage data only (no NWS thresholds)
- **Red** — no stage data at all

In [ ]:
# import plotly.express as px
# from IPython.display import HTML

# # Merge lat/lon into coverage
# map_df = coverage.merge(
#     site_info[["site_no", "latitude", "longitude", "station_name"]],
#     on="site_no", how="left"
# ).dropna(subset=["latitude", "longitude"])

# def _coverage_tier(row):
#     if not row["has_stage_data"]:
#         return "No stage data"
#     if not row["has_flood_stage"]:
#         return "Stage only"
#     return "Full coverage"

# map_df["tier"] = map_df.apply(_coverage_tier, axis=1)

# tier_order = ["Full coverage", "Stage only", "No stage data"]
# tier_colors = {
#     "Full coverage": "green",
#     "Stage only":    "orange",
#     "No stage data": "red",
# }

# # Append counts to tier labels
# counts = map_df["tier"].value_counts()
# label_map = {t: f"{t} (n={counts.get(t, 0):,})" for t in tier_order}
# map_df["tier"] = map_df["tier"].map(label_map)
# tier_order_labeled = [label_map[t] for t in tier_order]
# tier_colors_labeled = {label_map[t]: c for t, c in tier_colors.items()}

# print("Sites per tier:")
# print(map_df["tier"].value_counts().reindex(tier_order_labeled).to_string())

# fig = px.scatter_geo(
#     map_df,
#     lat="latitude",
#     lon="longitude",
#     color="tier",
#     category_orders={"tier": tier_order_labeled},
#     color_discrete_map=tier_colors_labeled,
#     hover_name="station_name",
#     hover_data={
#         "site_no": True,
#         "has_stage_data": True,
#         "has_flood_stage": True,
#         "has_reach_id": True,
#         "latitude": False,
#         "longitude": False,
#     },
#     scope="usa",
#     title="USGS Gage Data Coverage",
# )
# fig.update_traces(marker=dict(size=6, opacity=0.85))
# fig.update_layout(
#     legend_title_text="Data tier",
#     margin=dict(l=0, r=0, t=40, b=0),
#     height=600,
# )
# HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

## 3. Flood Stage Thresholds

In [ ]:
# flood_stages = pd.read_parquet(DATA_DIR / "metadata" / "flood_stages.parquet")
# print(f"Shape: {flood_stages.shape}")
# flood_stages

## 3b. Gage Map (USGS → NWS LID / NWM Reach)

In [ ]:
# gauge_map = pd.read_parquet(DATA_DIR / "metadata" / "gage_map.parquet")
# print(f"Shape: {gauge_map.shape}")
# print(f"Sites with reach_id: {gauge_map['reach_id'].notna().sum()} / {len(gauge_map)}")
# gauge_map

## 4. Streamflow

In [ ]:
# streamflow = pd.read_parquet(DATA_DIR / "streamflow" / "streamflow.parquet")

# print(f"Shape: {streamflow.shape}")
# print(f"Date range: {streamflow['date'].min().date()} → {streamflow['date'].max().date()}")
# print("\nRecords per site:")
# print(streamflow.groupby("site_no").size().rename("n_records").to_string())

In [ ]:
# print("First 5 rows:")
# display(streamflow.head())
# print("Last 5 rows:")
# display(streamflow.tail())

In [ ]:
# streamflow['discharge_cd'].unique()

## 5. Channel Geometry

In [ ]:
geometry = pd.read_parquet(DATA_DIR / "metadata" / "channel_geometry.parquet")
display(geometry)

In [ ]:
bankfull_equations = pd.unique(geometry['bkfw_equation'])
bankfull_equations

## 6. Specific Stream Power

In [ ]:
# power = pd.read_parquet(DATA_DIR / "metadata" / "stream_power.parquet")
# display(power)

## 7. Drainage Area

In [ ]:
da = geometry.merge(
    site_info[["site_no", "latitude", "longitude", "station_name", "drainage_area_sqmi"]],
    on='site_no', how='left'
)

In [ ]:
map_params = ['action_ssp_wm2', 'flood_ssp_wm2', 'moderate_ssp_wm2', 'major_ssp_wm2']

fig = plt.figure(figsize=(6.5, 4.5), dpi=150)

ax   = make_conus_ax(fig, pos=(1, 1, 1), title='Bankfull Width DA Limits')
sc   = ax.scatter(
    da['longitude'][da['drainage_area_sqmi'] < da['bkfw_da_min_sqmi']], da['latitude'][da['drainage_area_sqmi'] < da['bkfw_da_min_sqmi']],
    c='r',
    s=14, alpha=0.8, linewidths=0,
    transform=ccrs.PlateCarree(), zorder=4,
    label=f'DA < min (n={len(da['longitude'][da['drainage_area_sqmi'] < da['bkfw_da_min_sqmi']])})'
)

sc   = ax.scatter(
    da['longitude'][da['drainage_area_sqmi'] > da['bkfw_da_max_sqmi']], da['latitude'][da['drainage_area_sqmi'] > da['bkfw_da_max_sqmi']],
    c='b',
    s=14, alpha=0.8, linewidths=0,
    transform=ccrs.PlateCarree(), zorder=4,
    label=f'DA > max (n={len(da['longitude'][da['drainage_area_sqmi'] > da['bkfw_da_max_sqmi']])})'
)

# fig.suptitle(f'Spatial distribution of Specific Stream Power  |  n={len(map_df)}', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()